# **Cell 1 – Structired Directory Scraping Function**

In [ ]:
import os
import json
from pathlib import Path
from collections import defaultdict
import pandas as pd
from IPython.display import display

# ============================================================
#  CONFIG  – just change this one line
# ============================================================
ROOT = Path("/kaggle/input")          # or a more specific folder if you want
# ============================================================

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.gif', '.webp'}
META_EXTS  = {'.csv', '.tsv', '.xlsx', '.xls', '.json', '.txt', '.xml'}

structure = {}
meta_files = []

def is_image(p: Path) -> bool:
    return p.suffix.lower() in IMAGE_EXTS

def walk(path: Path, current_dict: dict):
    """Recursively build folder structure. Stop when an image is found."""
    print(f"Checking → {path}")          # ← live progress
    
    try:
        entries = sorted(path.iterdir())
    except PermissionError:
        print(f"  ⚠️  Permission denied: {path}")
        return

    has_image = any(is_image(e) for e in entries if e.is_file())
    
    if has_image:
        # This folder contains images → stop going deeper
        current_dict["__contains_images__"] = True
        current_dict["__subdirs__"] = [e.name for e in entries if e.is_dir()]
        print(f"  → Contains images. Stopping deeper traversal.")
        return

    # No images here → continue
    for entry in entries:
        if entry.is_dir():
            current_dict[entry.name] = {}
            walk(entry, current_dict[entry.name])
        elif entry.suffix.lower() in META_EXTS:
            meta_files.append(str(entry))
            current_dict.setdefault("__meta_files__", []).append(entry.name)
            print(f"  → Found metadata: {entry.name}")

# Start walking
print("Scanning structure (stopping at first image in any folder)...\n")
walk(ROOT, structure)

# Save JSON
json_path = "/kaggle/working/structure_inventory.json"
with open(json_path, "w") as f:
    json.dump(structure, f, indent=2)
print(f"\nJSON saved → {json_path}")

# Pretty print the tree
def print_tree(d, indent=0, max_depth=4):
    if indent > max_depth:
        print("  " * indent + "...")
        return
    for k, v in d.items():
        if k.startswith("__"):
            continue
        print("  " * indent + f"📁 {k}/")
        if isinstance(v, dict):
            print_tree(v, indent + 1, max_depth)

print("\n===== FOLDER STRUCTURE (stopped before images) =====")
print_tree(structure)

# Table of all metadata files found
print("\n===== METADATA / CSV FILES FOUND =====")
if meta_files:
    df_meta = pd.DataFrame({
        "Metadata File": meta_files,
        "Parent Folder": [str(Path(p).parent) for p in meta_files]
    })
    display(df_meta)
else:
    print("No CSV / metadata files found.")

print("\nDone. Structure inventory ready.")

# **Cell 2 – Master Function For Analysis**

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# MASTER ANALYSIS FUNCTION
# ============================================================
def analyze_dataset(dataset_name: str, root_path: str, class_folders: list = None, 
                    csv_path: str = None, label_col: str = None, image_col: str = None,
                    max_samples_per_class: int = 3):
    """
    Universal analyzer for fundus datasets in the 8-class group.
    
    Parameters:
    -----------
    dataset_name      : Nice display name
    root_path         : Path that contains the class folders OR the images folder
    class_folders     : List of class folder names (if folder-based). Leave None if CSV-based.
    csv_path          : Path to CSV (if label is in CSV)
    label_col         : Column name for class/label in CSV
    image_col         : Column name for image filename in CSV
    max_samples_per_class : How many sample images to show per class
    """
    
    print("="*80)
    print(f"DATASET: {dataset_name}")
    print("="*80)
    
    root = Path(root_path)
    image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
    
    # --------------------------------------------------------
    # 1. Collect images + labels
    # --------------------------------------------------------
    data = []          # list of (class_name, full_path)
    
    if csv_path is not None:
        # CSV-based dataset
        df = pd.read_csv(csv_path)
        print(f"CSV loaded: {csv_path}  |  shape = {df.shape}")
        print("Columns:", df.columns.tolist())
        
        if label_col is None or image_col is None:
            print("⚠️  Please provide label_col and image_col")
            return
        
        for _, row in df.iterrows():
            img_name = str(row[image_col])
            label = str(row[label_col])
            # try to find the image
            possible = list(root.rglob(img_name))
            if possible:
                data.append((label, possible[0]))
            else:
                # sometimes only filename without extension or different location
                possible = list(root.rglob(f"*{Path(img_name).stem}*"))
                if possible:
                    data.append((label, possible[0]))
    else:
        # Folder-based dataset
        if class_folders is None:
            # auto-detect class folders (immediate subdirs that contain images)
            class_folders = []
            for p in root.iterdir():
                if p.is_dir():
                    has_img = any(f.suffix.lower() in image_exts for f in p.rglob("*") if f.is_file())
                    if has_img:
                        class_folders.append(p.name)
        
        print(f"Class folders found: {class_folders}")
        
        for cls in class_folders:
            cls_path = root / cls
            if not cls_path.exists():
                # try recursive search
                matches = list(root.rglob(cls))
                if matches:
                    cls_path = matches[0]
                else:
                    print(f"⚠️  Class folder not found: {cls}")
                    continue
            
            for img_path in cls_path.rglob("*"):
                if img_path.suffix.lower() in image_exts:
                    data.append((cls, img_path))
    
    if len(data) == 0:
        print("❌ No images found. Check the path.")
        return
    
    # --------------------------------------------------------
    # 2. Build summary table
    # --------------------------------------------------------
    class_counts = Counter([d[0] for d in data])
    total_images = sum(class_counts.values())
    
    # Resolution & size statistics
    res_per_class = defaultdict(set)
    size_per_class = defaultdict(list)
    
    print("\nCollecting resolution & size info (this may take a moment)...")
    for cls, path in data:
        try:
            img = cv2.imread(str(path))
            if img is None:
                continue
            h, w = img.shape[:2]
            res_per_class[cls].add(f"{w}x{h}")
            size_mb = path.stat().st_size / (1024 * 1024)
            size_per_class[cls].append(size_mb)
        except Exception as e:
            pass
    
    rows = []
    for cls in sorted(class_counts.keys()):
        count = class_counts[cls]
        pct = count / total_images * 100
        resolutions = ", ".join(sorted(res_per_class[cls])) if res_per_class[cls] else "N/A"
        avg_size = np.mean(size_per_class[cls]) if size_per_class[cls] else 0
        rows.append({
            "Class": cls,
            "Image Count": count,
            "Percentage": f"{pct:.1f}%",
            "Resolution(s)": resolutions,
            "Avg Size (MB)": f"{avg_size:.3f}"
        })
    
    # Total row
    all_sizes = [s for sizes in size_per_class.values() for s in sizes]
    rows.append({
        "Class": "TOTAL",
        "Image Count": total_images,
        "Percentage": "100%",
        "Resolution(s)": "-",
        "Avg Size (MB)": f"{np.mean(all_sizes):.3f}" if all_sizes else "0"
    })
    
    df_summary = pd.DataFrame(rows)
    print("\n----- SUMMARY TABLE -----")
    display(df_summary)
    
    # --------------------------------------------------------
    # 3. Per-class sample images
    # --------------------------------------------------------
    print("\n----- SAMPLE IMAGES PER CLASS -----")
    
    classes = sorted(class_counts.keys())
    n_classes = len(classes)
    
    # Limit samples
    samples = defaultdict(list)
    for cls, path in data:
        if len(samples[cls]) < max_samples_per_class:
            samples[cls].append(path)
    
    # Plot
    cols = min(max_samples_per_class, 3)
    rows_plot = n_classes
    
    fig, axes = plt.subplots(rows_plot, cols, figsize=(4*cols, 3.5*rows_plot))
    if rows_plot == 1:
        axes = np.array([axes])
    if cols == 1:
        axes = axes.reshape(-1, 1)
    
    for i, cls in enumerate(classes):
        for j in range(cols):
            ax = axes[i, j]
            if j < len(samples[cls]):
                img = cv2.imread(str(samples[cls][j]))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                ax.imshow(img)
                ax.set_title(f"{cls}", fontsize=10)
            ax.axis("off")
    
    plt.suptitle(f"{dataset_name} – Sample Images", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()
    
    # --------------------------------------------------------
    # 4. Statistical analysis
    # --------------------------------------------------------
    print("\n----- IMAGE STATISTICAL ANALYSIS -----")
    
    all_widths, all_heights, all_sizes_mb = [], [], []
    channels = []
    
    for cls, path in data[:min(500, len(data))]:   # sample up to 500 for speed
        try:
            img = cv2.imread(str(path))
            if img is None:
                continue
            h, w = img.shape[:2]
            all_widths.append(w)
            all_heights.append(h)
            all_sizes_mb.append(path.stat().st_size / (1024*1024))
            channels.append(img.shape[2] if len(img.shape) == 3 else 1)
        except:
            pass
    
    stats = {
        "Total Images Analyzed": len(all_widths),
        "Unique Resolutions": len(set(zip(all_widths, all_heights))),
        "Width  – min / mean / max": f"{min(all_widths)} / {np.mean(all_widths):.0f} / {max(all_widths)}",
        "Height – min / mean / max": f"{min(all_heights)} / {np.mean(all_heights):.0f} / {max(all_heights)}",
        "File Size (MB) – min / mean / max": f"{min(all_sizes_mb):.3f} / {np.mean(all_sizes_mb):.3f} / {max(all_sizes_mb):.3f}",
        "Channels (most common)": Counter(channels).most_common(1)[0][0] if channels else "N/A"
    }
    
    for k, v in stats.items():
        print(f"{k:35s} : {v}")
    
    print("\n" + "="*80 + "\n")

# **Part 3 – Individual Calls**

In [ ]:
# 1. ODIR-5K (andrewmvd) - CORRECTED
analyze_dataset(
    dataset_name = "ODIR-5K",
    root_path    = "/kaggle/input/datasets/andrewmvd/ocular-disease-recognition-odir5k",
    csv_path     = "/kaggle/input/datasets/andrewmvd/ocular-disease-recognition-odir5k/full_df.csv",
    label_col    = "labels",      # we will confirm after loading
    image_col    = "filename"     # we will confirm after loading
)

In [ ]:
# 2. Combined Fundus Images - CORRECTED
analyze_dataset(
    dataset_name = "Combined Fundus Images",
    root_path    = "/kaggle/input/datasets/rohitrawat25/combined-fundus-images/Dataset/images",
    csv_path     = "/kaggle/input/datasets/rohitrawat25/combined-fundus-images/Dataset/label_images.csv",
    label_col    = "label",
    image_col    = "images"
)

In [ ]:
# 3. Ocular Disease Dataset (manan1717)
analyze_dataset(
    dataset_name = "Ocular Disease Dataset (manan1717)",
    root_path    = "/kaggle/input/datasets/manan1717/ocular-disease-dataset/preprocessed",
    class_folders = ["A", "C", "D", "G", "H", "M", "N"]
)

In [ ]:
# 4. eye-diseases-classification
analyze_dataset(
    dataset_name = "eye-diseases-classification",
    root_path    = "/kaggle/input/datasets/fahadhasan93/eye-diseases-classification/dataset",
    class_folders = ["cataract", "diabetic_retinopathy", "glaucoma", "normal"]
)

In [ ]:
# 5. Eye Diseases (belalsafy)
analyze_dataset(
    dataset_name = "Eye Diseases (belalsafy)",
    root_path    = "/kaggle/input/datasets/belalsafy/eye-diseases/dataset",
    class_folders = [
        "Age related Macular Degeneration", "Cataract", "Diabetes",
        "Glaucoma", "Hypertension", "Normal", "Other diseases", "Pathological Myopia"
    ]
)

In [ ]:
# 6. eye diseases (mohamedgamal295)
analyze_dataset(
    dataset_name = "eye diseases (mohamedgamal295)",
    root_path    = "/kaggle/input/datasets/mohamedgamal295/eye-diseases/dataset",
    class_folders = ["cataract", "diabetic_retinopathy", "glaucoma", "normal", "retina_disease"]
)

In [ ]:
# 7. eye diseases 2
analyze_dataset(
    dataset_name = "eye diseases 2",
    root_path    = "/kaggle/input/datasets/mohamedgamal295/eye-diseases-2/eye diseases",
    class_folders = ["cataract", "diabetic_retinopathy", "glaucoma", "normal"]
)

In [ ]:
# 8. Eye diseases (aealemran)
analyze_dataset(
    dataset_name = "Eye diseases (aealemran)",
    root_path    = "/kaggle/input/datasets/aealemran/eye-diseases/Eye diseases",
    class_folders = [
        "Color Fundus", "Diabetic Retinopathy", "Disc Edema", "Glaucoma",
        "Healthy", "Macular Scar", "Myopia", "Pterygium",
        "Retinal Detachment", "Retinitis Pigmentosa"
    ]
)

In [ ]:
# 9. Eye-fundus10 (train)
analyze_dataset(
    dataset_name = "Eye-fundus10 (train)",
    root_path    = "/kaggle/input/datasets/ermano123/eye-fundus10/eye/train",
    class_folders = None
)

In [ ]:
# 10. Eye Disease Detection (train)
analyze_dataset(
    dataset_name = "Eye Disease Detection (train)",
    root_path    = "/kaggle/input/datasets/h100gpu/eye-disease-detection/Dataset/train",
    class_folders = ["amd", "cataract", "diabetes", "glaucoma", "normal"]
)

In [ ]:
# 11. Ocular Fundus Image For Disease Detection
analyze_dataset(
    dataset_name = "Ocular Fundus Image For Disease Detection",
    root_path    = "/kaggle/input/datasets/mdeconozzamanecon/ocular-disease/augmented",
    class_folders = ["Retinoblastoma", "cataract", "glaucoma", "normal"]
)

In [ ]:
# 12. Processed Eye Dataset V2 (Augmented)
analyze_dataset(
    dataset_name = "Processed Eye Dataset V2 (Augmented)",
    root_path    = "/kaggle/input/datasets/mohammadsadman21/processed-eye-dataset-v2/Eye_Disease_Images_dataset/Augmented",
    class_folders = None
)

In [ ]:
# 13. preprocessed_eye_diseases_fundus_images
analyze_dataset(
    dataset_name = "preprocessed_eye_diseases_fundus_images",
    root_path    = "/kaggle/input/datasets/gunavenkatdoddi/preprocessed-eye-diseases-fundus-images/seg_dataset",
    class_folders = ["cataract", "diabetic_retinopathy", "glaucoma", "normal"]
)

In [ ]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime

# ============================================================
# 1. PATHS DATABASE
# ============================================================
paths_db = {
    "group": "fundus-8class-odir",
    "created": datetime.now().isoformat(),
    "datasets": {
        "ODIR-5K": {
            "kaggle_slug": "andrewmvd/ocular-disease-recognition-odir5k",
            "root": "/kaggle/input/datasets/andrewmvd/ocular-disease-recognition-odir5k",
            "csv": "/kaggle/input/datasets/andrewmvd/ocular-disease-recognition-odir5k/full_df.csv",
            "images_folder": "/kaggle/input/datasets/andrewmvd/ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Training Images",
            "label_col": "labels",
            "image_col": "filename",
            "type": "csv_based",
            "classes": ["N", "D", "G", "C", "A", "H", "M", "O"]
        },
        "Combined Fundus Images": {
            "kaggle_slug": "rohitrawat25/combined-fundus-images",
            "root": "/kaggle/input/datasets/rohitrawat25/combined-fundus-images/Dataset",
            "csv": "/kaggle/input/datasets/rohitrawat25/combined-fundus-images/Dataset/label_images.csv",
            "images_folder": "/kaggle/input/datasets/rohitrawat25/combined-fundus-images/Dataset/images",
            "label_col": "label",
            "image_col": "images",
            "type": "csv_based",
            "classes": ["N", "D", "G", "C", "A", "H", "M", "O"]
        },
        "Ocular Disease Dataset (manan1717)": {
            "kaggle_slug": "manan1717/ocular-disease-dataset",
            "root": "/kaggle/input/datasets/manan1717/ocular-disease-dataset/preprocessed",
            "csv": None,
            "images_folder": "/kaggle/input/datasets/manan1717/ocular-disease-dataset/preprocessed",
            "type": "folder_based",
            "classes": ["A", "C", "D", "G", "H", "M", "N"]
        },
        "eye-diseases-classification": {
            "kaggle_slug": "fahadhasan93/eye-diseases-classification",
            "root": "/kaggle/input/datasets/fahadhasan93/eye-diseases-classification/dataset",
            "csv": None,
            "images_folder": "/kaggle/input/datasets/fahadhasan93/eye-diseases-classification/dataset",
            "type": "folder_based",
            "classes": ["cataract", "diabetic_retinopathy", "glaucoma", "normal"]
        },
        "Eye Diseases (belalsafy)": {
            "kaggle_slug": "belalsafy/eye-diseases",
            "root": "/kaggle/input/datasets/belalsafy/eye-diseases/dataset",
            "csv": None,
            "images_folder": "/kaggle/input/datasets/belalsafy/eye-diseases/dataset",
            "type": "folder_based",
            "classes": [
                "Age related Macular Degeneration", "Cataract", "Diabetes",
                "Glaucoma", "Hypertension", "Normal", "Other diseases", "Pathological Myopia"
            ]
        },
        "eye diseases (mohamedgamal295)": {
            "kaggle_slug": "mohamedgamal295/eye-diseases",
            "root": "/kaggle/input/datasets/mohamedgamal295/eye-diseases/dataset",
            "csv": None,
            "images_folder": "/kaggle/input/datasets/mohamedgamal295/eye-diseases/dataset",
            "type": "folder_based",
            "classes": ["cataract", "diabetic_retinopathy", "glaucoma", "normal", "retina_disease"]
        },
        "eye diseases 2": {
            "kaggle_slug": "mohamedgamal295/eye-diseases-2",
            "root": "/kaggle/input/datasets/mohamedgamal295/eye-diseases-2/eye diseases",
            "csv": None,
            "images_folder": "/kaggle/input/datasets/mohamedgamal295/eye-diseases-2/eye diseases",
            "type": "folder_based",
            "classes": ["cataract", "diabetic_retinopathy", "glaucoma", "normal"]
        },
        "Eye diseases (aealemran)": {
            "kaggle_slug": "aealemran/eye-diseases",
            "root": "/kaggle/input/datasets/aealemran/eye-diseases/Eye diseases",
            "csv": None,
            "images_folder": "/kaggle/input/datasets/aealemran/eye-diseases/Eye diseases",
            "type": "folder_based",
            "classes": [
                "Color Fundus", "Diabetic Retinopathy", "Disc Edema", "Glaucoma",
                "Healthy", "Macular Scar", "Myopia", "Pterygium",
                "Retinal Detachment", "Retinitis Pigmentosa"
            ]
        },
        "Eye-fundus10": {
            "kaggle_slug": "ermano123/eye-fundus10",
            "root": "/kaggle/input/datasets/ermano123/eye-fundus10/eye",
            "csv": None,
            "images_folder": "/kaggle/input/datasets/ermano123/eye-fundus10/eye/train",
            "type": "folder_based",
            "classes": [
                "Central Serous Chorioretinopathy", "Diabetic Retinopathy", "Disc Edema",
                "Glaucoma", "Healthy", "Macular Scar", "Myopia", "Pterygium",
                "Retinal Detachment", "Retinitis Pigmentosa"
            ]
        },
        "Eye Disease Detection": {
            "kaggle_slug": "h100gpu/eye-disease-detection",
            "root": "/kaggle/input/datasets/h100gpu/eye-disease-detection/Dataset",
            "csv": None,
            "images_folder": "/kaggle/input/datasets/h100gpu/eye-disease-detection/Dataset/train",
            "type": "folder_based",
            "classes": ["amd", "cataract", "diabetes", "glaucoma", "normal"]
        },
        "Ocular Fundus Image": {
            "kaggle_slug": "mdeconozzamanecon/ocular-disease",
            "root": "/kaggle/input/datasets/mdeconozzamanecon/ocular-disease/augmented",
            "csv": None,
            "images_folder": "/kaggle/input/datasets/mdeconozzamanecon/ocular-disease/augmented",
            "type": "folder_based",
            "classes": ["Retinoblastoma", "cataract", "glaucoma", "normal"]
        },
        "Processed Eye Dataset V2": {
            "kaggle_slug": "mohammadsadman21/processed-eye-dataset-v2",
            "root": "/kaggle/input/datasets/mohammadsadman21/processed-eye-dataset-v2/Eye_Disease_Images_dataset/Augmented",
            "csv": None,
            "images_folder": "/kaggle/input/datasets/mohammadsadman21/processed-eye-dataset-v2/Eye_Disease_Images_dataset/Augmented",
            "type": "folder_based",
            "classes": [
                "Central Serous Chorioretinopathy", "Diabetic Retinopathy", "Disc Edema",
                "Glaucoma", "Healthy", "Macular Scar", "Myopia", "Pterygium",
                "Retinal Detachment", "Retinitis Pigmentosa"
            ]
        },
        "preprocessed_eye_diseases_fundus_images": {
            "kaggle_slug": "gunavenkatdoddi/preprocessed-eye-diseases-fundus-images",
            "root": "/kaggle/input/datasets/gunavenkatdoddi/preprocessed-eye-diseases-fundus-images/seg_dataset",
            "csv": None,
            "images_folder": "/kaggle/input/datasets/gunavenkatdoddi/preprocessed-eye-diseases-fundus-images/seg_dataset",
            "type": "folder_based",
            "classes": ["cataract", "diabetic_retinopathy", "glaucoma", "normal"]
        }
    }
}

# Save paths DB
with open("/kaggle/working/paths_db.json", "w") as f:
    json.dump(paths_db, f, indent=2)
print("✅ paths_db.json saved")

# ============================================================
# 2. SUMMARY LOGS DB
# All 13 datasets filled in from the analyze_dataset() output tables above
# (keys match paths_db exactly, so the master table join below now works for every row)
# ============================================================
summary_db = {
    "group": "fundus-8class-odir",
    "created": datetime.now().isoformat(),
    "results": {
        "ODIR-5K": {
            "total_images": 6392,
            "classes": {"N": 2873, "D": 1608, "O": 708, "C": 293, "G": 284, "A": 266, "M": 232, "H": 128},
            "resolution": "512x512",
            "avg_size_mb": 0.059,
            "notes": "labels column contains string lists like ['X']"
        },
        "Combined Fundus Images": {
            "total_images": 9868,
            "classes": {"N": 3230, "D": 1608, "G": 1071, "M": 1036, "C": 897, "O": 888, "A": 708, "H": 430},
            "resolution": "512x512",
            "avg_size_mb": 0.11,
            "notes": "labels column contains string lists like ['X']"
        },
        "Ocular Disease Dataset (manan1717)": {
            "total_images": 10449,
            "classes": {"D": 3128, "N": 2997, "G": 1646, "C": 1008, "M": 739, "A": 511, "H": 420},
            "resolution": "Variable (22+ unique; 512x512 to 1266x1265)",
            "avg_size_mb": 0.376
        },
        "eye-diseases-classification": {
            "total_images": 4217,
            "classes": {"diabetic_retinopathy": 1098, "normal": 1074, "cataract": 1038, "glaucoma": 1007},
            "resolution": "Variable (5 unique; 256x256 to 2592x1728)",
            "avg_size_mb": 0.175
        },
        "Eye Diseases (belalsafy)": {
            "total_images": 8230,
            "classes": {"Normal": 2280, "Diabetes": 2256, "Other diseases": 1958, "Glaucoma": 430, "Cataract": 424, "Pathological Myopia": 348, "Age related Macular Degeneration": 328, "Hypertension": 206},
            "resolution": "Variable (15+ unique; 1280x960 to 1677x1472)",
            "avg_size_mb": 0.193
        },
        "eye diseases (mohamedgamal295)": {
            "total_images": 4317,
            "classes": {"diabetic_retinopathy": 1098, "normal": 1074, "cataract": 1038, "glaucoma": 1007, "retina_disease": 100},
            "resolution": "Variable (5 unique; 256x256 to 2592x1728)",
            "avg_size_mb": 0.237
        },
        "eye diseases 2": {
            "total_images": 4217,
            "classes": {"diabetic_retinopathy": 1098, "normal": 1074, "cataract": 1038, "glaucoma": 1007},
            "resolution": "Variable (5 unique; 256x256 to 2592x1728)",
            "avg_size_mb": 0.175,
            "notes": "identical class counts/sizes to 'eye diseases (mohamedgamal295)' \u2014 likely the same underlying data"
        },
        "Eye diseases (aealemran)": {
            "total_images": 9160,
            "classes": {"Diabetic Retinopathy": 2047, "Glaucoma": 1588, "Healthy": 1468, "Myopia": 1244, "Macular Scar": 1033, "Retinitis Pigmentosa": 476, "Disc Edema": 450, "Retinal Detachment": 446, "Color Fundus": 348, "Pterygium": 60},
            "resolution": "640x640",
            "avg_size_mb": 0.027
        },
        "Eye-fundus10": {
            "total_images": 11364,
            "classes": {"Diabetic Retinopathy": 2410, "Glaucoma": 2015, "Healthy": 1873, "Myopia": 1575, "Macular Scar": 1355, "Retinitis Pigmentosa": 583, "Disc Edema": 533, "Retinal Detachment": 525, "Central Serous Chorioretinopathy": 424, "Pterygium": 71},
            "resolution": "224x224",
            "avg_size_mb": 0.006
        },
        "Eye Disease Detection": {
            "total_images": 1989,
            "classes": {"normal": 410, "diabetes": 400, "cataract": 400, "amd": 394, "glaucoma": 385},
            "resolution": "Variable (20+ unique; 512x512 to 1285x1396)",
            "avg_size_mb": 0.243
        },
        "Ocular Fundus Image": {
            "total_images": 4952,
            "classes": {"Retinoblastoma": 1238, "cataract": 1238, "glaucoma": 1238, "normal": 1238},
            "resolution": "Variable (11+ unique; 130x130 to 1848x2592)",
            "avg_size_mb": 1.258
        },
        "Processed Eye Dataset V2": {
            "total_images": 16242,
            "classes": {"Diabetic Retinopathy": 3444, "Glaucoma": 2880, "Healthy": 2676, "Myopia": 2251, "Macular Scar": 1937, "Retinitis Pigmentosa": 834, "Disc Edema": 762, "Retinal Detachment": 750, "Central Serous Chorioretinopathy": 606, "Pterygium": 102},
            "resolution": "512x512",
            "avg_size_mb": 0.034
        },
        "preprocessed_eye_diseases_fundus_images": {
            "total_images": 4217,
            "classes": {"diabetic_retinopathy": 1098, "normal": 1074, "cataract": 1038, "glaucoma": 1007},
            "resolution": "Variable (5 unique; 256x256 to 2592x1728)",
            "avg_size_mb": 0.14
        }

    }
}

with open("/kaggle/working/summary_db.json", "w") as f:
    json.dump(summary_db, f, indent=2)
print("✅ summary_db.json saved")

# ============================================================
# 3. MASTER CSV TABLE
# ============================================================
master_rows = []

for name, info in paths_db["datasets"].items():
    master_rows.append({
        "Dataset": name,
        "Kaggle Slug": info["kaggle_slug"],
        "Type": info["type"],
        "Root Path": info["root"],
        "CSV Path": info.get("csv", ""),
        "Images Folder": info["images_folder"],
        "Label Column": info.get("label_col", ""),
        "Image Column": info.get("image_col", ""),
        "Classes": " | ".join(info["classes"]),
        "Num Classes": len(info["classes"]),
        "Total Images": summary_db["results"].get(name, {}).get("total_images", ""),
        "Resolution": summary_db["results"].get(name, {}).get("resolution", ""),
        "Avg Size (MB)": summary_db["results"].get(name, {}).get("avg_size_mb", ""),
        "Per-Class Counts": " | ".join(
            f"{cls}:{cnt}" for cls, cnt in summary_db["results"].get(name, {}).get("classes", {}).items()
        )
    })

df_master = pd.DataFrame(master_rows)
df_master.to_csv("/kaggle/working/master_table.csv", index=False)
print("✅ master_table.csv saved")

print("\n===== MASTER TABLE =====")
display(df_master)

print("\nFiles created in /kaggle/working/ :")
print(" - paths_db.json")
print(" - summary_db.json")
print(" - master_table.csv")